# Unity Catalog Hands-On Examples

This notebook demonstrates common Unity Catalog patterns in Databricks:

- creating a catalog and schema
- creating managed tables
- querying fully qualified table names
- granting permissions
- inspecting objects with SQL

## Setup notes

These examples assume your Databricks environment is configured with Unity Catalog and that your account has privileges to create catalogs, schemas, and tables.

In [ ]:
catalog_name = "demo_governance"
schema_name = "curated"
table_name = "customer_orders"
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

print(full_table_name)

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

print(f"Ensured catalog and schema exist: {catalog_name}.{schema_name}")

## Create a managed table

Managed tables store both metadata and managed data under the control of the platform.

In [ ]:
orders = [
    (1, "north", 1200.0, "completed"),
    (2, "south", 800.0, "completed"),
    (3, "west", 400.0, "pending")
]

orders_df = spark.createDataFrame(
    orders,
    ["order_id", "region", "amount", "status"]
)

orders_df.write.mode("overwrite").saveAsTable(full_table_name)
print(f"Wrote table {full_table_name}")

In [ ]:
display(spark.table(full_table_name).orderBy("order_id"))

## Use fully qualified names

Using catalog.schema.table makes object references explicit and avoids ambiguity across environments.

In [ ]:
result_df = spark.sql(f"""
SELECT region, COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM {full_table_name}
GROUP BY region
ORDER BY total_amount DESC
""")

display(result_df)

## Grants example

Privileges are typically granted to groups rather than individual users. Replace the placeholder principal with a real group in your environment.

In [ ]:
principal = "analyst_group"

spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog_name} TO `{principal}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog_name}.{schema_name} TO `{principal}`")
spark.sql(f"GRANT SELECT ON TABLE {full_table_name} TO `{principal}`")

print(f"Granted catalog, schema, and table access to {principal}")

In [ ]:
display(spark.sql(f"SHOW GRANTS ON TABLE {full_table_name}"))

## Discover objects

These commands help users understand what exists inside the catalog and schema.

In [ ]:
display(spark.sql(f"SHOW CATALOGS LIKE '{catalog_name}'"))
display(spark.sql(f"SHOW SCHEMAS IN {catalog_name}"))
display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}"))

## Production notes

- Use catalogs to separate business domains or governance boundaries
- Use schemas to separate layers like raw, curated, and serving
- Grant access to groups, not individual users
- Prefer fully qualified names in jobs and production code
- Keep development and production data in clearly separated namespaces